AREA-WEIGHTED MONTHLY SST AND CHL (GEOGRAPHIC OCEAN MASK) extraction

In [ ]:
# ======================================================
# AREA-WEIGHTED MONTHLY SST AND CHL (GEOGRAPHIC OCEAN MASK)
# Same mask logic as the CHL coverage audit: Natural Earth land polygons,
# independent of data validity. Study area: coastal Bangladesh / NE Bay of Bengal.
# Output: full monthly calendar (Aug-2002 to Jan-2026), NaN where no valid data.
# ======================================================

import os
import numpy as np
import pandas as pd
import xarray as xr

try:
    import cartopy.feature as cfeature
    from shapely.geometry import box
    from shapely.ops import unary_union
except ImportError:
    raise ImportError("Install the required packages first:  pip install cartopy shapely")

# ======================================================
# USER INPUTS
# ======================================================
SST_FILE = "SST_Complete_to_Jan2026.nc"
CHL_FILE = "CHL_Complete_to_Jan2026.nc"
OUTPUT_FOLDER = "Data"
OUTPUT_NAME = "Area_Weighted_Monthly_SST_CHL_masked.xlsx"

# Study area and period
LON_MIN, LON_MAX = 89.0, 92.5
LAT_MIN, LAT_MAX = 20.5, 22.8
START_DATE = "2002-08-01"
END_DATE = "2026-01-01"

# Minimum share of ocean cells with valid data for a month to be used.
# None = use any month that has at least one valid ocean cell.
# Choose a value after reading the Threshold_Test sheet, and before looking
# at any ENSO/IOD relationship. State it in the Methods.
MIN_COVERAGE_PCT = 20

THRESHOLDS = [5, 10, 15, 20, 25, 30, 40, 50, 60]   # for the Threshold_Test sheet
LOW_COVERAGE_REPORT_PCT = 20                        # months below this are listed

# Variable and coordinate names in each file
SST_CFG = dict(name="SST", path=SST_FILE, var="sst", lat="lat", lon="lon", time="time",
               kelvin_check=True)
CHL_CFG = dict(name="CHL", path=CHL_FILE, var="CHL", lat="latitude", lon="longitude", time="time",
               kelvin_check=False)
# ======================================================


# ------------------------------------------------------
# 1. Natural Earth land geometry (clipped to the study area for speed)
# ------------------------------------------------------
def build_land_geometry():
    print("Loading Natural Earth land polygons (10m)...")
    feature = cfeature.NaturalEarthFeature(category="physical", name="land",
                                           scale="10m", facecolor="none")
    land_union = unary_union(list(feature.geometries()))
    clip_box = box(LON_MIN - 1, LAT_MIN - 1, LON_MAX + 1, LAT_MAX + 1)
    try:
        return land_union.intersection(clip_box)
    except Exception:
        return land_union.buffer(0).intersection(clip_box)


def points_on_land(land_geom, lon2d, lat2d):
    """True where the cell centre lies on land (same rule as the audit script)."""
    try:
        import shapely
        return np.asarray(shapely.contains_xy(land_geom, lon2d, lat2d))     # shapely >= 2.0
    except (ImportError, AttributeError):
        pass
    try:
        from shapely import vectorized
        return np.asarray(vectorized.contains(land_geom, lon2d, lat2d))     # shapely 1.8
    except (ImportError, AttributeError):
        from shapely.geometry import Point
        from shapely.prepared import prep
        prepared = prep(land_geom)
        out = np.zeros(lon2d.shape, dtype=bool)
        for idx in np.ndindex(lon2d.shape):
            out[idx] = prepared.contains(Point(float(lon2d[idx]), float(lat2d[idx])))
        return out


# ------------------------------------------------------
# 2. Extract one variable: mask, area-weighted mean, coverage
# ------------------------------------------------------
def extract(cfg, land_geom, calendar):
    name, t, la, lo = cfg["name"], cfg["time"], cfg["lat"], cfg["lon"]
    print(f"\n=========== {name} ===========")

    ds = xr.open_dataset(cfg["path"])
    if cfg["var"] not in ds.data_vars:
        raise KeyError(f"Variable '{cfg['var']}' not in {cfg['path']}. Found: {list(ds.data_vars)}")
    da = ds[cfg["var"]]
    for c in (t, la, lo):
        if c not in da.dims:
            raise KeyError(f"Dimension '{c}' not found for {name}. Dimensions: {da.dims}")

    # Drop extra singleton dimensions (for example depth or altitude)
    for d in [d for d in da.dims if d not in (t, la, lo)]:
        if da.sizes[d] != 1:
            raise ValueError(f"{name}: extra dimension '{d}' has size {da.sizes[d]}; select one level first.")
        da = da.isel({d: 0}, drop=True)

    # Put every time stamp on the first day of its month
    idx = da.indexes[t]
    if not isinstance(idx, pd.DatetimeIndex):
        idx = idx.to_datetimeindex()
    month_start = idx.to_period("M").to_timestamp()
    if month_start.duplicated().any():
        raise ValueError(f"{name}: more than one time step in the same month. Average them first.")
    da = da.assign_coords({t: month_start}).astype("float64")

    # Ascending order, then study area and period
    da = da.sortby(t).sortby(la).sortby(lo)
    da = da.sel({la: slice(LAT_MIN, LAT_MAX), lo: slice(LON_MIN, LON_MAX),
                 t: slice(START_DATE, END_DATE)})
    print(f"Grid: {da.sizes[la]} lat x {da.sizes[lo]} lon | months in file (study period): {da.sizes[t]}")

    # Geographic ocean mask on this variable's own grid
    lon2d, lat2d = np.meshgrid(da[lo].values, da[la].values)
    is_land = points_on_land(land_geom, lon2d, lat2d)
    ocean = xr.DataArray(~is_land, coords={la: da[la].values, lo: da[lo].values}, dims=[la, lo])
    n_total, n_ocean = int(ocean.size), int(ocean.sum())
    n_land = n_total - n_ocean
    print(f"Cells: total {n_total} | geographic ocean {n_ocean} | geographic land {n_land}")

    da_m = da.where(ocean).load()

    # Unit and value checks (nothing is changed except the Kelvin conversion)
    if cfg["kelvin_check"] and float(da_m.mean(skipna=True)) > 100:
        print("SST looks like Kelvin: converting to °C (subtract 273.15).")
        da_m = da_m - 273.15
    if name == "CHL":
        n_bad = int((da_m <= 0).sum())
        if n_bad:
            print(f"WARNING: {n_bad} CHL values are zero or negative. They were NOT removed.")

    # Area-weighted mean: cos(latitude) weights over valid ocean cells only
    weights = np.cos(np.deg2rad(da_m[la]))
    mean = da_m.weighted(weights).mean(dim=[la, lo])
    valid = da_m.notnull().sum(dim=[la, lo])

    res = pd.DataFrame({"raw_mean": mean.values, "valid_cells": valid.values.astype(int)},
                       index=pd.DatetimeIndex(da[t].values))
    in_file = calendar.isin(res.index)
    res = res.reindex(calendar)
    res["valid_cells"] = res["valid_cells"].fillna(0).astype(int)
    res["ocean_cells"] = n_ocean
    res["coverage_pct"] = res["valid_cells"] / n_ocean * 100
    res["in_file"] = in_file

    ok = res["raw_mean"].notna()
    if MIN_COVERAGE_PCT is not None:
        ok &= res["coverage_pct"] >= MIN_COVERAGE_PCT
    res["used"] = ok
    res["used_mean"] = res["raw_mean"].where(ok)

    cov = res.loc[res["in_file"], "coverage_pct"]
    print(f"Coverage (months in file): min {cov.min():.1f}% | mean {cov.mean():.1f}% | max {cov.max():.1f}%")
    print(f"Months used: {int(ok.sum())} of {len(calendar)} | months without a value: {int((~ok).sum())}")

    grid = dict(Variable=name, Latitudes=da.sizes[la], Longitudes=da.sizes[lo],
                Total_cells=n_total, Ocean_cells=n_ocean, Land_cells=n_land,
                Ocean_pct=n_ocean / n_total * 100)
    ds.close()
    return res, grid


# ------------------------------------------------------
# 3. Run
# ------------------------------------------------------
calendar = pd.date_range(START_DATE, END_DATE, freq="MS")
land_geom = build_land_geometry()

sst_tbl, sst_grid = extract(SST_CFG, land_geom, calendar)
chl_tbl, chl_grid = extract(CHL_CFG, land_geom, calendar)

# ------------------------------------------------------
# 4. Output tables
# ------------------------------------------------------
month_year = calendar.strftime("%b-%Y")

series = pd.DataFrame({
    "Month-Year": month_year,
    "SST (°C)": sst_tbl["used_mean"].values,
    "CHL (mg/m^3)": chl_tbl["used_mean"].values,
})

audit = pd.DataFrame({"Month-Year": month_year, "Date": calendar})
for tag, tbl in (("SST", sst_tbl), ("CHL", chl_tbl)):
    audit[f"{tag}_in_file"] = tbl["in_file"].values
    audit[f"{tag}_ocean_cells"] = tbl["ocean_cells"].values
    audit[f"{tag}_valid_cells"] = tbl["valid_cells"].values
    audit[f"{tag}_coverage_pct"] = tbl["coverage_pct"].values
    audit[f"{tag}_mean_before_threshold"] = tbl["raw_mean"].values
    audit[f"{tag}_used"] = tbl["used"].values

rows = []
for tag, tbl in (("SST", sst_tbl), ("CHL", chl_tbl)):
    has = tbl["raw_mean"].notna()
    rows.append({"Variable": tag, "Coverage_threshold_pct": "any valid cell",
                 "Months_retained": int(has.sum()), "Months_excluded": int((~has).sum())})
    for thr in THRESHOLDS:
        kept = has & (tbl["coverage_pct"] >= thr)
        rows.append({"Variable": tag, "Coverage_threshold_pct": thr,
                     "Months_retained": int(kept.sum()),
                     "Months_excluded": int(len(calendar) - kept.sum())})
threshold_table = pd.DataFrame(rows)

low = []
for tag, tbl in (("SST", sst_tbl), ("CHL", chl_tbl)):
    sel = tbl["coverage_pct"] < LOW_COVERAGE_REPORT_PCT
    for d, r in tbl[sel].iterrows():
        low.append({"Variable": tag, "Month-Year": d.strftime("%b-%Y"),
                    "In_file": bool(r["in_file"]), "Valid_cells": int(r["valid_cells"]),
                    "Ocean_cells": int(r["ocean_cells"]), "Coverage_pct": r["coverage_pct"]})
low_table = pd.DataFrame(low)

both = series["SST (°C)"].notna() & series["CHL (mg/m^3)"].notna()
summary = pd.DataFrame({
    "Item": [
        "Study area (lat, lon)", "Study period", "Calendar months",
        "Weights", "Mask", "Minimum coverage threshold",
        "SST months with a value", "CHL months with a value", "Months with both",
        "SST ocean cells / land cells", "CHL ocean cells / land cells",
    ],
    "Value": [
        f"{LAT_MIN}-{LAT_MAX} N, {LON_MIN}-{LON_MAX} E",
        f"{month_year[0]} to {month_year[-1]}", len(calendar),
        "cos(latitude), valid ocean cells only",
        "Natural Earth 10m land polygons, applied on each variable's own grid",
        "none (any valid cell)" if MIN_COVERAGE_PCT is None else f">= {MIN_COVERAGE_PCT}% of ocean cells",
        int(series["SST (°C)"].notna().sum()), int(series["CHL (mg/m^3)"].notna().sum()),
        int(both.sum()),
        f"{sst_grid['Ocean_cells']} / {sst_grid['Land_cells']}",
        f"{chl_grid['Ocean_cells']} / {chl_grid['Land_cells']}",
    ],
})

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
output_file = os.path.join(OUTPUT_FOLDER, OUTPUT_NAME)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    series.to_excel(writer, sheet_name="Monthly_Series", index=False)
    audit.to_excel(writer, sheet_name="Coverage_Audit", index=False)
    threshold_table.to_excel(writer, sheet_name="Threshold_Test", index=False)
    low_table.to_excel(writer, sheet_name="Low_Coverage_Months", index=False)
    pd.DataFrame([sst_grid, chl_grid]).to_excel(writer, sheet_name="Grid_and_Mask", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)

# ------------------------------------------------------
# 5. Console summary
# ------------------------------------------------------
print("\n=========== FINAL SERIES ===========")
print(f"Calendar months: {len(calendar)} | SST: {int(series['SST (°C)'].notna().sum())} | "
      f"CHL: {int(series['CHL (mg/m^3)'].notna().sum())} | both: {int(both.sum())}")
print("\nMonths without a CHL value:")
print(", ".join(series.loc[series["CHL (mg/m^3)"].isna(), "Month-Year"]) or "none")
print("\nMonths without an SST value:")
print(", ".join(series.loc[series["SST (°C)"].isna(), "Month-Year"]) or "none")
print("\nThreshold test:")
print(threshold_table.to_string(index=False))
print("\nSaved:", output_file)

De-seasonalize, trend checking and detrending ( if applicable)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# ============================================================
# 1. File paths
# ============================================================

input_file = "Area_Weighted_Monthly_SST_CHL_masked.xlsx"

output_file = "Anomalized_detrended_SST&CHL.xlsx"


# ============================================================
# 2. Read Excel file
# ============================================================

df = pd.read_excel(input_file)

# Clean extra spaces in column names
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

print("Columns found:")
print(list(df.columns))


# ============================================================
# 3. Column names
# ============================================================

date_col = "Month-Year"
nino_col = "Niño 3.4"
dmi_col = "DMI"
chl_col = "CHL (mg/m^3)"
sst_col = "SST (°C)"


# ============================================================
# 4. Parse date and sort chronologically
# ============================================================

df["Date"] = pd.to_datetime(
    df[date_col],
    format="%b-%Y"
)

df = df.sort_values("Date").reset_index(drop=True)


# ============================================================
# 5. Check the study period and missing CHL values
# ============================================================

print("\nStudy period:")
print("Start:", df["Date"].min().strftime("%b-%Y"))
print("End  :", df["Date"].max().strftime("%b-%Y"))

print("\nTotal number of calendar months:", len(df))

print("Missing CHL months:", df[chl_col].isna().sum())
print("Missing SST months:", df[sst_col].isna().sum())

if df[chl_col].isna().sum() > 0:
    print("\nCHL missing months:")
    print(
        df.loc[df[chl_col].isna(), date_col]
        .tolist()
    )


# ============================================================
# 6. Extract calendar month
# ============================================================

df["Month"] = df["Date"].dt.month


# ============================================================
# 7. Monthly climatology
#    NaN values are automatically ignored by mean()
# ============================================================

sst_climatology = df.groupby("Month")[sst_col].mean()
chl_climatology = df.groupby("Month")[chl_col].mean()


print("\nMonthly climatology calculated.")


# ============================================================
# 8. Remove seasonal cycle
#
#    CHL and SST are deseasonalized.
#    Niño 3.4 and DMI remain unchanged.
#
#    NaN CHL months remain NaN.
# ============================================================

df[sst_col] = (
    df[sst_col]
    - df["Month"].map(sst_climatology)
)

df[chl_col] = (
    df[chl_col]
    - df["Month"].map(chl_climatology)
)


print("Seasonal cycle removed from CHL and SST.")


# ============================================================
# 9. Create actual calendar-month time index
#
#    This is important because 13 CHL observations are missing.
#    The time index represents actual elapsed calendar months.
# ============================================================

df["t"] = (
    (df["Date"].dt.year - df["Date"].dt.year.min()) * 12
    + (df["Date"].dt.month - df["Date"].dt.month.min())
)


# ============================================================
# 10. Trend test function
#
#     NaN values are excluded from the trend calculation.
# ============================================================

def trend_test(series, time):

    # Keep only valid observations
    valid = series.notna() & time.notna()

    y = series.loc[valid]
    t = time.loc[valid]

    N = len(y)

    # Ordinary linear regression
    slope, intercept, r_value, p_naive, std_err = stats.linregress(
        t,
        y
    )

    # Lag-1 autocorrelation
    r1 = y.autocorr(lag=1)

    # Effective sample size
    if pd.notna(r1) and r1 > 0:
        Neff = N * (1 - r1) / (1 + r1)
    else:
        Neff = N

    # Corrected standard error
    se_corrected = std_err * np.sqrt(N / Neff)

    # Corrected t-statistic
    t_stat = slope / se_corrected

    # Effective degrees of freedom
    df_eff = max(Neff - 2, 1)

    # Corrected p-value
    p_corrected = 2 * (
        1 - stats.t.cdf(
            abs(t_stat),
            df=df_eff
        )
    )

    return {
        "N": N,
        "Slope (per month)": slope,
        "Slope (per year)": slope * 12,
        "Slope (per decade)": slope * 120,
        "Intercept": intercept,
        "R-squared": r_value ** 2,
        "Lag-1 autocorr (r1)": r1,
        "Effective N (Neff)": Neff,
        "p-value (naive)": p_naive,
        "p-value (corrected)": p_corrected,
    }


# ============================================================
# 11. Trend testing and conditional detrending
#
#     SST and CHL only.
#
#     If corrected p < 0.05:
#         -> significant trend
#         -> detrending applied
#
#     If corrected p >= 0.05:
#         -> no significant trend
#         -> original deseasonalized series retained
# ============================================================

summary_rows = []

for var in [sst_col, chl_col]:

    res = trend_test(
        df[var],
        df["t"]
    )

    significant = (
        res["p-value (corrected)"] < 0.05
    )

    print("\n========================================")
    print("Variable:", var)
    print("========================================")

    print(
        "Valid observations (N):",
        res["N"]
    )

    print(
        "Slope per year:",
        round(res["Slope (per year)"], 6)
    )

    print(
        "Slope per decade:",
        round(res["Slope (per decade)"], 6)
    )

    print(
        "Lag-1 autocorrelation:",
        round(res["Lag-1 autocorr (r1)"], 6)
    )

    print(
        "Effective sample size:",
        round(res["Effective N (Neff)"], 6)
    )

    print(
        "Naive p-value:",
        round(res["p-value (naive)"], 6)
    )

    print(
        "Corrected p-value:",
        round(res["p-value (corrected)"], 6)
    )


    # --------------------------------------------------------
    # Detrend only if corrected trend is significant
    # --------------------------------------------------------

    if significant:

        print(
            "Significant trend detected "
            "-> Detrending applied."
        )

        trend_line = (
            res["Intercept"]
            + res["Slope (per month)"] * df["t"]
        )

        # Only valid observations are changed.
        # Existing NaN values remain NaN.
        df[var] = df[var] - trend_line

        detrending_applied = "Yes"

    else:

        print(
            "No significant trend detected "
            "-> No detrending applied."
        )

        detrending_applied = "No"


    # --------------------------------------------------------
    # Save trend information
    # --------------------------------------------------------

    row = {
        "Variable": var,
        "N": res["N"],
        "Slope (per month)": res["Slope (per month)"],
        "Slope (per year)": res["Slope (per year)"],
        "Slope (per decade)": res["Slope (per decade)"],
        "Intercept": res["Intercept"],
        "R-squared": res["R-squared"],
        "Lag-1 autocorr (r1)": res["Lag-1 autocorr (r1)"],
        "Effective N (Neff)": res["Effective N (Neff)"],
        "p-value (naive)": res["p-value (naive)"],
        "p-value (corrected)": res["p-value (corrected)"],
        "Significant (p<0.05)": (
            "Yes" if significant else "No"
        ),
        "Detrending applied": detrending_applied
    }

    summary_rows.append(row)


# ============================================================
# 12. Trend summary table
# ============================================================

summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df.drop(
    columns=["Intercept"]
)

summary_df = summary_df.round(6)


# ============================================================
# 13. Restore Month-Year format
# ============================================================

df["Month-Year"] = (
    df["Date"].dt.strftime("%b-%Y")
)


# ============================================================
# 14. Final output table
#
#     Niño 3.4 and DMI remain unchanged.
#     CHL missing months remain NaN.
# ============================================================

final_output = df[
    [
        "Month-Year",
        nino_col,
        dmi_col,
        chl_col,
        sst_col
    ]
]


# ============================================================
# 15. Final NaN check
# ============================================================

print("\n========================================")
print("FINAL DATA CHECK")
print("========================================")

print(
    "Total months:",
    len(final_output)
)

print(
    "Missing CHL:",
    final_output[chl_col].isna().sum()
)

print(
    "Missing SST:",
    final_output[sst_col].isna().sum()
)

print(
    "Missing Niño 3.4:",
    final_output[nino_col].isna().sum()
)

print(
    "Missing DMI:",
    final_output[dmi_col].isna().sum()
)


# ============================================================
# 16. Save both sheets
# ============================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    final_output.to_excel(
        writer,
        sheet_name="Detrended_Data",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Trend_Summary",
        index=False
    )


# ============================================================
# 17. Completion message
# ============================================================

print("\n========================================")
print("PROCESSING COMPLETE")
print("========================================")

print(
    "1. Monthly climatology calculated."
)

print(
    "2. CHL and SST deseasonalized."
)

print(
    "3. Autocorrelation-adjusted trend tests performed."
)

print(
    "4. Detrending applied only where corrected p < 0.05."
)

print(
    "5. Niño 3.4 and DMI remained unchanged."
)

print(
    "6. Missing CHL observations were retained as NaN."
)

print(
    "\nOutput saved to:"
)

print(output_file)

Columns found:
['Month-Year', 'Niño 3.4', 'DMI', 'CHL (mg/m^3)', 'SST (°C)']

Study period:
Start: Aug-2002
End  : Jan-2026

Total number of calendar months: 282
Missing CHL months: 13
Missing SST months: 0

CHL missing months:
['Jun-2003', 'May-2012', 'Jun-2012', 'Jul-2012', 'Aug-2012', 'May-2013', 'Jun-2014', 'Aug-2014', 'Jun-2015', 'Jul-2015', 'Aug-2015', 'Dec-2015', 'Apr-2016']

Monthly climatology calculated.
Seasonal cycle removed from CHL and SST.

Variable: SST (°C)
Valid observations (N): 282
Slope per year: 0.028087
Slope per decade: 0.280871
Lag-1 autocorrelation: 0.388682
Effective sample size: 124.140359
Naive p-value: 0.0
Corrected p-value: 0.000114
Significant trend detected -> Detrending applied.

Variable: CHL (mg/m^3)
Valid observations (N): 269
Slope per year: -0.001849
Slope per decade: -0.018491
Lag-1 autocorrelation: 0.632525
Effective sample size: 60.550873
Naive p-value: 0.833356
Corrected p-value: 0.920749
No significant trend detected -> No detrending applied.

Z score standardization of Nino 3.4, DMI, SST and CHL 

In [2]:
import pandas as pd

# ============================================================
# FILE PATHS
# ============================================================

input_file = "Anomalized_detrended_SST&CHL.xlsx" 

output_file = "Standardized_SST_CHL_ENSO_IOD_282months.xlsx"


# ============================================================
# READ DATA
# ============================================================

df = pd.read_excel(input_file)

# Clean extra spaces in column names
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

date_col = "Month-Year"

cols = [
    "Niño 3.4",
    "DMI",
    "SST (°C)",
    "CHL (mg/m^3)"
]


# ============================================================
# FORCE NUMERIC
# ============================================================

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")


# ============================================================
# BASIC CHECKS
# ============================================================

print("\nDataset information")
print("-------------------")
print("Number of rows:", len(df))
print("Date range:", df[date_col].iloc[0], "to", df[date_col].iloc[-1])

print("\nMissing values:")
print(df[cols].isna().sum())


# ============================================================
# CHECK THAT ONLY CHL HAS THE EXPECTED 13 NaNs
# ============================================================

expected_chl_missing = 13

for c in cols:

    n_missing = df[c].isna().sum()

    if c == "CHL (mg/m^3)":

        if n_missing != expected_chl_missing:
            raise ValueError(
                f"Unexpected number of CHL missing values: "
                f"{n_missing}. Expected {expected_chl_missing}."
            )

    else:

        if n_missing > 0:
            raise ValueError(
                f"Unexpected missing values found in {c}: "
                f"{n_missing} values."
            )


# ============================================================
# STANDARDIZATION
# ============================================================
#
# Z = (X - mean) / standard deviation
#
# ddof=1 is retained to match your previous 269-month analysis.
#
# NaN values are NOT filled or removed.
# Pandas mean() and std() automatically ignore NaN values.
# Therefore CHL is standardized using its 269 available months.
# ============================================================

df_std = df.copy()

params = []

for c in cols:

    # Calculate scaling parameters from available observations
    mean_value = df[c].mean()
    std_value = df[c].std(ddof=1)

    # Prevent division by zero
    if pd.isna(std_value) or std_value == 0:
        raise ValueError(
            f"Invalid standard deviation for {c}: {std_value}"
        )

    # Standardize while preserving NaN positions
    df_std[c] = (df[c] - mean_value) / std_value

    # Save scaling parameters
    params.append({
        "Variable": c,
        "N used for standardization": df[c].notna().sum(),
        "N missing": df[c].isna().sum(),
        "Mean (original)": mean_value,
        "Std (original)": std_value
    })


# ============================================================
# VERIFY STANDARDIZATION
# ============================================================

print("\nStandardization check")
print("---------------------")

for c in cols:

    valid = df_std[c].dropna()

    print(
        f"{c}: "
        f"N={len(valid)}, "
        f"mean={valid.mean():.10f}, "
        f"SD={valid.std(ddof=1):.10f}"
    )


# ============================================================
# VERIFY CHL NaNs WERE PRESERVED
# ============================================================

original_chl_nan = df["CHL (mg/m^3)"].isna()
standardized_chl_nan = df_std["CHL (mg/m^3)"].isna()

if not original_chl_nan.equals(standardized_chl_nan):
    raise ValueError(
        "CHL NaN positions changed during standardization."
    )

print(
    "\nCHL missing values preserved:",
    standardized_chl_nan.sum()
)


# ============================================================
# SAVE OUTPUT
# ============================================================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df_std.to_excel(
        writer,
        sheet_name="Standardized_Data",
        index=False
    )

    pd.DataFrame(params).round(6).to_excel(
        writer,
        sheet_name="Scaling_Parameters",
        index=False
    )


print("\nSaved to:")
print(output_file)


Dataset information
-------------------
Number of rows: 282
Date range: Aug-2002 to Jan-2026

Missing values:
Niño 3.4         0
DMI              0
SST (°C)         0
CHL (mg/m^3)    13
dtype: int64

Standardization check
---------------------
Niño 3.4: N=282, mean=0.0000000000, SD=1.0000000000
DMI: N=282, mean=0.0000000000, SD=1.0000000000
SST (°C): N=282, mean=-0.0000000000, SD=1.0000000000
CHL (mg/m^3): N=269, mean=0.0000000000, SD=1.0000000000

CHL missing values preserved: 13

Saved to:
E:\Regional_Studies_Marine\Revision\Dataset_for_lag_test\Standardized_SST_CHL_ENSO_IOD_282months.xlsx
